In [19]:
import json
import numpy as np
import cv2
from typing import List
from rl_carcassone.env.game.utils import PixelMeaning

In [20]:
with open(r"C:\Users\cherniak\pet_projects\carcassonne\rl_carcassone\env\game\assets\cards_raw.json") as f:
    cards = json.load(f)

In [21]:
rotation_to_new_order = {
    90: [20, 15, 10, 5, 0, 21, 16, 11, 6, 1, 22, 17, 12, 7, 2, 23, 18, 13, 8, 3, 24, 19, 14, 9, 4],
    180: [24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0],
    270: [4, 9, 14, 19, 24, 3, 8, 13, 18, 23, 2, 7, 12, 17, 22, 1, 6, 11, 16, 21, 0, 5, 10, 15, 20],
}


def rotate_values(values: str, orientation: int) -> str:
    if orientation == 0:
        return values
    elif orientation == 90:
        return "".join([values[i] for i in rotation_to_new_order[90]])
    elif orientation == 180:
        return "".join([values[i] for i in rotation_to_new_order[180]])
    elif orientation == 270:
        return "".join([values[i] for i in rotation_to_new_order[270]])
    else:
        raise ValueError


def rotate_properties(properties: List[int], orientation: int) -> List[int]:
    if orientation == 0:
        return properties
    elif orientation == 90:
        return [properties[i] for i in rotation_to_new_order[90]]
    elif orientation == 180:
        return [properties[i] for i in rotation_to_new_order[180]]
    elif orientation == 270:
        return [properties[i] for i in rotation_to_new_order[270]]
    else:
        raise ValueError

In [22]:
def get_connectors(property_index: int, property_indices: List[int]) -> List[str]:
    connectors = []
    used_cells = np.where(np.array(property_indices) == property_index)[0]
    if 2 in used_cells:
        connectors.append("N")
    else:
        if 1 in used_cells:
            connectors.append("NW")
        if 3 in used_cells:
            connectors.append("NE")
    if 22 in used_cells:
        connectors.append("S")
    else:
        if 21 in used_cells:
            connectors.append("SW")
        if 23 in used_cells:
            connectors.append("SE")
    if 10 in used_cells:
        connectors.append("W")
    else:
        if 5 in used_cells:
            connectors.append("WN")
        if 15 in used_cells:
            connectors.append("WS")
    if 14 in used_cells:
        connectors.append("E")
    else:
        if 9 in used_cells:
            connectors.append("EN")
        if 19 in used_cells:
            connectors.append("ES")
    return connectors

In [23]:
cards_updated = []

for ref_card in cards:
    card_updated_data = {
        "type": ref_card["type"],
        "count": ref_card["count"],
        "shield": ref_card.get("shield", False),
        "options": dict(),
    }
    for ref_orientation in ref_card["rotations"]:
        rotated_values = rotate_values(ref_card["values"], ref_orientation)
        rotated_properties = rotate_properties(ref_card["properties"], ref_orientation)
        option = {
            "possible_neighbors": {"N": [], "S": [], "W": [], "E": []},
            "values": rotated_values,
            "properties": rotated_properties,
            "properties_metas": [],
        }
        for property_index, property_type in enumerate(ref_card["property_types"]):
            connectors = get_connectors(property_index, rotated_properties)
            property_meta = {
                "type": property_type,
                "connectors": connectors,
            }
            option["properties_metas"].append(property_meta)
        for candidate_card in cards:
            for candidate_orientation in candidate_card["rotations"]:
                rotated_candidate_values = rotate_values(candidate_card["values"], candidate_orientation)
                if rotated_values[2] == rotated_candidate_values[22]:
                    option["possible_neighbors"]["N"].append((candidate_card["type"], candidate_orientation))
                if rotated_values[22] == rotated_candidate_values[2]:
                    option["possible_neighbors"]["S"].append((candidate_card["type"], candidate_orientation))
                if rotated_values[10] == rotated_candidate_values[14]:
                    option["possible_neighbors"]["W"].append((candidate_card["type"], candidate_orientation))
                if rotated_values[14] == rotated_candidate_values[10]:
                    option["possible_neighbors"]["E"].append((candidate_card["type"], candidate_orientation))
        card_updated_data["options"][ref_orientation] = option
    cards_updated.append(card_updated_data)

In [24]:
save_fname = r"C:\Users\cherniak\pet_projects\carcassonne\rl_carcassone\env\game\assets\cards.json"
with open(save_fname, "w") as f:
    json.dump(cards_updated, f, ensure_ascii=True, indent=4)